# Module 11 — Production (Python view)

Two things you can usefully script from Python in a production-shaped environment:
1. Apply RLS context per request.
2. Drive the bouncer-friendly client settings (no client-side prepared statements when behind transaction-pooling).

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
DSN = f"postgresql://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}@{os.environ['PGHOST']}:{os.environ['PGPORT']}/{os.environ['PGDATABASE']}"
print('OK')

## 1. Apply RLS context per request

Pattern: open a transaction, `SET LOCAL app.user_id`, run the user's queries, commit. The setting is scoped to the transaction so concurrent users do not see each other's context.

In [ ]:
import psycopg

def run_as_user(user_id: int, sql: str):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute("SET LOCAL app.user_id = %s", (str(user_id),))
                cur.execute(sql)
                return cur.fetchall()

# Note: RLS is currently DISABLED on app.posts at the end of lab.sql. To see
# the effect, re-enable with:
#   ALTER TABLE app.posts ENABLE ROW LEVEL SECURITY;
# then re-run this cell.
print(run_as_user(7, 'SELECT count(*) FROM app.posts'))

## 2. Behind PgBouncer in transaction mode

Disable client-side prepared statements so server rotation does not break your driver.

In [ ]:
# psycopg 3
with psycopg.connect(DSN, prepare_threshold=None) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT 1')
        print(cur.fetchone())

# asyncpg (illustrative)
# pool = await asyncpg.create_pool(DSN, statement_cache_size=0)

## 3. Operational queries from Python

Scriptable health checks you might add to a service's /healthz or to a cron probe.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
engine = create_engine(os.environ['DATABASE_URL'])

queries = {
    'long_xacts': """
        SELECT pid, usename, state, now() - xact_start AS age, left(query, 80) AS q
        FROM pg_stat_activity
        WHERE xact_start IS NOT NULL AND now() - xact_start > interval '1 minute'
    """,
    'bloat_top': """
        SELECT relname, n_dead_tup,
               round(100.0*n_dead_tup / nullif(n_live_tup + n_dead_tup, 0), 2) AS dead_pct
        FROM pg_stat_user_tables WHERE schemaname='app'
        ORDER BY dead_pct DESC NULLS LAST LIMIT 5
    """,
    'db_size': 'SELECT pg_size_pretty(pg_database_size(current_database())) AS db_size'
}
with engine.connect() as conn:
    for k, q in queries.items():
        print(f'\n## {k}')
        print(pd.read_sql(text(q), conn))